# 01 — VLM Room Analysis API

**Goal:** Use Qwen2-VL-2B-Instruct to inspect an uploaded room image and return a validated `RoomAnalysis` JSON object. The final section exposes the same function through a FastAPI endpoint for the main application.

## Execution order
1. Check the runtime and prepare dependencies.
2. Authenticate to Hugging Face if the model requires access.
3. Load Qwen2-VL and its processor.
4. Define the output schema and JSON parser.
5. Implement and test room analysis.
6. Start the FastAPI service and expose it through ngrok.

**Runtime:** Google Colab with a GPU recommended. Run cells from top to bottom. Do not put API tokens directly in notebook code; store them in Colab Secrets.

## 1. Runtime and dependency diagnostics
These checks are diagnostic. If imports already work, avoid reinstalling packages unnecessarily.

In [1]:
import torch
print("torch:", torch.__version__)
try:
    import torchaudio
    print("torchaudio:", torchaudio.__version__)
except Exception as e:
    print("torchaudio issue:", e)

torch: 2.11.0+cu130
torchaudio: 2.11.0+cu130


In [2]:
import importlib.util
spec = importlib.util.find_spec("torchaudio")
print("find_spec result:", spec)

import subprocess
result = subprocess.run(["pip", "show", "torchaudio"], capture_output=True, text=True)
print("pip show output:", result.stdout, result.stderr)

find_spec result: ModuleSpec(name='torchaudio', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7b2a26f093d0>, origin='/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torchaudio'])
pip show output: Name: torchaudio
Version: 2.11.0+cu130
Summary: An audio package for PyTorch
Home-page: https://github.com/pytorch/audio
Author: Soumith Chintala, David Pollack, Sean Naren, Peter Goldsborough, Moto Hira, Caroline Chen, Jeff Hwang, Zhaoheng Ni, Xiaohui Zhang
Author-email: soumith@pytorch.org
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: 
 


In [3]:
!pip install torchaudio -q

## 2. Hugging Face authentication
If the model is gated in your environment, authenticate with your Hugging Face account. Keep access tokens private.

In [4]:
from huggingface_hub import login

login()

## 3. Load Qwen2-VL-2B-Instruct
The compatibility shim below was used in this Colab environment to work around a `torchaudio` import issue. Keep it only if the current runtime still needs it.

In [5]:
import sys
import types
import importlib.machinery

fake_torchaudio = types.ModuleType("torchaudio")
fake_torchaudio.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
sys.modules["torchaudio"] = fake_torchaudio

from transformers.models.qwen2_vl.modeling_qwen2_vl import Qwen2VLForConditionalGeneration
from transformers import AutoProcessor
import torch

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256*28*28,
    max_pixels=768*28*28
)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 4. Define the structured output schema
The Pydantic model defines the fields returned by the room-analysis stage. The parser converts model output into validated data.

In [6]:
from pydantic import BaseModel
import json
import re

class RoomAnalysis(BaseModel):
    room_type: str
    furniture: list[str]
    decor: list[str]
    colors: list[str]
    style: str
    lighting: str
    spatial_observations: list[str]

def parse_room_analysis(raw_text: str) -> RoomAnalysis:
    cleaned = re.sub(r"^```(json)?\s*|\s*```$", "", raw_text.strip())
    data = json.loads(cleaned)
    return RoomAnalysis(**data)

## 5. Implement room-image analysis
The function asks the VLM for JSON, decodes its response, and validates it against `RoomAnalysis`.

In [7]:
from PIL import Image
import gc

def analyze_room(image: Image.Image) -> RoomAnalysis:
    structured_prompt = """Analyze this room and respond with ONLY a valid JSON object, no other text, in exactly this format:

{
  "room_type": "string",
  "furniture": ["item1", "item2"],
  "decor": ["item1", "item2"],
  "colors": ["color1", "color2"],
  "style": "string",
  "lighting": "string",
  "spatial_observations": ["observation1", "observation2"]
}

"furniture" is functional items (seating, tables, storage). "decor" is non-furniture items (art, mirrors, plants). "spatial_observations" describes positions and layout, e.g. "bookcase on left wall", "door on right side".

Do not include any explanation, markdown formatting, or text outside the JSON object."""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": structured_prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=250)

    result = processor.batch_decode(output, skip_special_tokens=True)
    raw_output = result[0].split("assistant\n")[-1]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return parse_room_analysis(raw_output)

## 6. Test analysis with an uploaded image
Upload one room image when prompted. The image is resized if its longest edge exceeds 1024 pixels to reduce memory use.

In [8]:
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

max_dim = 1024
if max(image.size) > max_dim:
    ratio = max_dim / max(image.size)
    image = image.resize((int(image.size[0]*ratio), int(image.size[1]*ratio)))

result = analyze_room(image)
print(result)

Saving room.jpg to room.jpg


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


room_type='Bedroom' furniture=['bed', 'nightstand', 'cabinet'] decor=['paintings', 'headboard'] colors=['beige', 'brown'] style='Classic' lighting='table lamps' spatial_observations=['bed against wall', 'nightstand on left side', 'cabinet on right side']


## 7. Create the FastAPI endpoint
The `/analyze` endpoint accepts an uploaded image and returns the validated room analysis.

In [9]:
!pip install fastapi uvicorn python-multipart nest-asyncio pyngrok -q

In [10]:
from fastapi import FastAPI, UploadFile, File
from io import BytesIO
from PIL import Image
import traceback

vlm_app = FastAPI(title="Interior Design VLM API")

In [11]:
@vlm_app.post("/analyze")
async def analyze_endpoint(file: UploadFile = File(...)):
    try:
        image_bytes = await file.read()

        image = Image.open(
            BytesIO(image_bytes)
        ).convert("RGB")

        print("Image received:", image.size)

        # resize to cap memory usage before running the VLM
        max_dim = 1024
        if max(image.size) > max_dim:
            ratio = max_dim / max(image.size)
            image = image.resize((int(image.size[0] * ratio), int(image.size[1] * ratio)))
            print("Image resized to:", image.size)

        result = analyze_room(image)

        print("VLM result:", result)

        return result.model_dump()

    except Exception as e:
        error_text = traceback.format_exc()

        print("========== VLM ERROR ==========")
        print(error_text)
        print("===============================")

        return {
            "status": "error",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "traceback": error_text
        }

## 8. Start the API server
This starts Uvicorn in a background thread inside the notebook runtime. Keep the Colab session running while the backend uses this service.

In [14]:
!pip install pyngrok -q

In [15]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_server():
    uvicorn.run(vlm_app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

## 9. Expose the API with ngrok
Add `NGROK_AUTHTOKEN` to Colab Secrets before running this cell. **Do not hardcode tokens in notebooks or commit them to GitHub.**

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
NGROK_TUNNEL = ngrok.connect(8000)
VLM_PUBLIC_URL = NGROK_TUNNEL.public_url

print("VLM API base URL:", VLM_PUBLIC_URL)
print("Interactive API docs:", f"{VLM_PUBLIC_URL}/docs")

## 10. Verify the running service

Open the printed `/docs` URL to inspect the API. The `/analyze` endpoint expects a **POST** request with an image file; a browser GET request to `/analyze` is not a valid test.

Example request from another Python process (replace the URL with the current printed URL):

In [ ]:
# Run this from a separate client environment, not necessarily in the server notebook.
# Replace VLM_PUBLIC_URL with the current ngrok URL printed above.
#
# import requests
# with open("room.jpg", "rb") as image_file:
#     response = requests.post(
#         f"{VLM_PUBLIC_URL}/analyze",
#         files={"file": ("room.jpg", image_file, "image/jpeg")},
#         timeout=180,
#     )
# response.raise_for_status()
# print(response.json())

## Troubleshooting notes
- If model loading fails, inspect the first dependency error before reinstalling packages.
- If CUDA memory is exhausted, restart the runtime and reduce `max_pixels` or the input image size.
- The public ngrok URL can change whenever the tunnel restarts; use the URL printed by the current session.
- Never share Hugging Face or ngrok tokens in screenshots, notebook outputs, or Git history.